# Lab 18 · Optimisation 3: make the model small, for real

**Time:** 55 min (25 before lunch, 30 after) · **Work in teams of 3**

**Where we are in the day:** Lab 17 gave us an accurate model (86 to 92 per cent exact with a one-line prompt), but it still takes about 3,000 MB.
**Today's question:** how small can we make the model, running on a CPU with no GPU, before its answers get worse?
**What you will do:**
- Merge the adapter into the model and convert it into one GGUF file.
- Shrink the file to four smaller sizes and run each size on the CPU only.
- Compare file size, speed, accuracy and perplexity, and choose a format.

**What you will see:** in our test run the file fell from 3,094 MB to 986 MB (Q4_K_M) with the same 83.3 per cent on our sample of 24 messages, and the smallest size (Q2_K) fell to 45.8 per cent.
**Why it matters on a device:** a device has a small memory and often no GPU, so the file size and the CPU speed decide whether the assistant can live there.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- The cells take about **8 minutes** in all (the CPU runs of 18.5 are most of it; each format has a 90 second limit).
- If a cell takes longer than that, wait until the time is over, then interrupt the kernel and ask the trainer.
- Before you start: the **setup script** (`setup_day3.sh`) must have finished (see 18.1), and Lab 17 must have been run. If your Lab 17 did not finish, the lab uses a prepared adapter. The scoreboard should already have two rows.

**Words used in this lab:**
- **Quantisation:** rounding every number of the model to fewer levels, like measuring in millimetres, centimetres or whole metres. Fewer levels need fewer **bits**: a bit is one yes-or-no digit, so 4 bits give 16 levels and 8 bits give 256.
- **Merge:** adding the sticky notes (the LoRA adapter) into the big matrices once, so that one ordinary model remains.
- **GGUF:** a single file format for running small models on laptops and devices, with or without quantisation.
- **Perplexity:** how surprised the model is by the correct answer. 1 means never surprised; higher means more surprised.

A 16-bit weight takes 2 bytes, a 4-bit weight takes half a byte. The model gets smaller and often faster, and the answers can get worse. **How much worse is the whole question, and we measure it.**

**Python in this lab:** loops, lists and dictionaries again. The conversion and the CPU runs are in `edge_gguf.py`.

## Part 1: make the files
Check the setup, merge the adapter, convert to GGUF and make the four smaller sizes (18.1 to 18.4). Mostly waiting for files to be written.

In [ ]:
import sys, os
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import torch
import pandas as pd
import edge_lab as el
import edge_nlu as nlu
from slm_lab import check, OUT
print("scoreboard so far:", list(el.load_scoreboard().keys()))

## 18.1 Check the setup (once)
This lab needs the **llama.cpp** project (it holds the converter script) and small Python packages to convert and run GGUF files on the CPU. The setup script `setup_day3.sh` that you started at the beginning of the day installs them. The cell below only checks.

In [ ]:
llama_folder_ok = os.path.exists("/workspace/llama.cpp")
try:
    import llama_cpp
    llama_package_ok = True
except ImportError:
    llama_package_ok = False
check(llama_folder_ok and llama_package_ok, "Setup check: llama.cpp and the GGUF tools are installed.",
      "Not installed yet. Run  bash setup_day3.sh  in a terminal, and wait for SETUP DONE at the end of /workspace/day3_setup.log. Then run this cell again.")

**You should see** a green line. If you see the warning instead, follow the instruction in it and ask the trainer if the log shows an error.

## 18.2 Merge the adapter into the model
**If your Lab 17 training did not finish, do not wait:** the next cell uses a **prepared adapter** that was trained in exactly the same way, so you can continue.

In Lab 17 the model and the LoRA adapter were **two parts**: the big book and the sticky notes. For serving, it is simpler to **merge** them once: the notes are written into the book. Then there is **one ordinary model** and no extra step for every request. *In one sentence: after the merge, the adapter is gone and the model is a normal one.*

> **Optional: if you are curious.** In the formula, `W` becomes `W + (alpha/r)·B·A`.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import shutil
ADAPTER_FOLDER = str(OUT / "edge_adapter")
MERGED_FOLDER = str(OUT / "edge_merged")
PREPARED_FOLDER = os.path.join(os.path.dirname(os.path.abspath(el.__file__)), "data", "edge_adapter_prepared")
if os.path.exists(ADAPTER_FOLDER):
    print("using your own adapter from Lab 17")
else:
    shutil.copytree(PREPARED_FOLDER, ADAPTER_FOLDER)          # safety net: Lab 17 did not finish, so use the prepared adapter
    print("Lab 17 left no adapter, so the PREPARED adapter is used (it was trained the same way)")

Now load the model, attach the adapter and merge.

In [ ]:
tok = AutoTokenizer.from_pretrained(el.MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(el.MODEL_ID, dtype=torch.bfloat16, device_map="auto")
with_adapter = PeftModel.from_pretrained(base, ADAPTER_FOLDER)
merged = with_adapter.merge_and_unload()
merged.save_pretrained(MERGED_FOLDER)
tok.save_pretrained(MERGED_FOLDER)
print("merged model saved:", MERGED_FOLDER, " weights:", el.model_size_mb(merged), "MB")

**You should see** the merged model, about 3,000 MB. Is it **exactly** the same model as before the merge? Rounding in 16-bit numbers can change the last digits. Let us check on the first 20 test messages. Write down how many replies are identical.

In [ ]:
tok.padding_side = "left"
golden = nlu.load_golden()
texts = []
for case in golden[:20]:
    texts.append(el.make_prompt_text(tok, nlu.SHORT_SYSTEM, case["text"]))
tok2, separate = el.load_edge_model(adapter=ADAPTER_FOLDER)
with_adapter_replies = el.ask_many(tok2, separate, texts)["replies"]
merged_replies = el.ask_many(tok, merged, texts)["replies"]
same = 0
for index in range(20):
    if with_adapter_replies[index] == merged_replies[index]:
        same = same + 1
print("identical replies:", same, "of 20")

**You should see** that **most** replies are identical (in our test run: 20 of 20) and that a few may differ in small ways. Merging gives a model that is **very nearly**, but **not guaranteed bit for bit**, the same. This is why we always test the merged model, never assume.

In [ ]:
del base, with_adapter, merged, separate
torch.cuda.empty_cache()

In [ ]:
check(os.path.exists(MERGED_FOLDER), "Checkpoint: the merged model is saved.",
      "The merged folder is missing. Run the merge cells of 18.2 again, in order.")

## 18.3 Convert to GGUF
**Predict.** This model has about 1.54 billion weights. On paper, write the size in megabytes at 16, 8, 5, 4 and 2 bits per weight, using *size = number of weights × bits ÷ 8* (a megabyte is a million bytes). Five numbers. The next cell prints the estimate, so you can check yourself.

In [ ]:
PARAMETERS = 1_540_000_000
for bits in [16, 8, 5, 4, 2]:
    print(bits, "bits per weight ->", round(PARAMETERS * bits / 8 / 1e6), "MB")

Now the real thing. The converter writes one 16-bit `.gguf` file.

In [ ]:
import edge_gguf as eg
GGUF_FOLDER = "/root/edge_gguf"                  # the pod's fast local disk: these files can be made again any time
os.makedirs(GGUF_FOLDER, exist_ok=True)
f16_file = os.path.join(GGUF_FOLDER, "intent-f16.gguf")
size = eg.convert_to_gguf(MERGED_FOLDER, f16_file)
print("16-bit GGUF:", size, "MB")

In [ ]:
check(os.path.exists(f16_file), "Checkpoint: the 16-bit GGUF file exists.",
      "The file is missing. Check that /workspace/llama.cpp exists (18.1), then run the conversion cell again.")

**You should see** about 3,100 MB, as predicted for 16 bits (in our test run: 3,094 MB).

## 18.4 Shrink it: four sizes
Four **levels** of quantisation. The names tell you the idea: `Q8_0` is about 8 bits per weight, `Q5_K_M` about 5, `Q4_K_M` about 4 and `Q2_K` about 2 to 3. *In one sentence: the smaller the number, the coarser the rounding.*

> **Optional: if you are curious.** The `K` formats are smarter: they use more bits for the weights that matter more.

In [ ]:
LEVELS = ["Q8_0", "Q5_K_M", "Q4_K_M", "Q2_K"]
files = {"F16": f16_file}
sizes = [{"format": "F16", "size MB": size}]
for level in LEVELS:
    path = os.path.join(GGUF_FOLDER, "intent-" + level + ".gguf")
    mb = eg.quantise(f16_file, path, level)
    files[level] = path
    sizes.append({"format": level, "size MB": mb})
pd.DataFrame(sizes)

In [ ]:
check(len(files) == 5, "Checkpoint: you have 5 GGUF files (F16 and four smaller sizes).",
      "Run the cell of 18.4 again. If a size fails, check that there is free disk space and ask the trainer.")

**You should see** the sizes fall step by step. In our test run: F16 3,094 MB, Q8_0 1,647, Q5_K_M 1,125, Q4_K_M 986 and Q2_K 676 MB. The low-bit files are somewhat larger than your estimate, because the K formats keep some parts in more bits. The smallest is under a quarter of the 16-bit size. **Nobody has checked the quality yet.** That is the next part.

## Part 2: run them and decide
**You now have:** five GGUF files, from 3,094 MB down to 676 MB, and nobody has yet checked their quality. Now we run each one on the CPU and decide. This part is the longest wait: read the next section while it runs.

## 18.5 Run it on the CPU only
We open each file **on the CPU**: no GPU, and only **4 threads** (4 workers at the same time), like a small device. A device answers one person at a time, so we send the messages one by one. We use a **sample of 24 of the 72 test messages** (every third one) so the run is short. One practice message is sent first and not counted, because the first message of a new file is always slow. The model answers with the one-line prompt it was trained with. Each format has a **time limit of 90 seconds**: if it is slower, the lab stops it and scores what is done.

In [ ]:
sample = []
for index in range(0, len(golden), 3):
    sample.append(golden[index])
print("sample:", len(sample), "messages")
print("reading the file once (a cold file is slow the first time):", eg.warm_up(files["Q4_K_M"]), "seconds")
llm = eg.load_gguf(files["Q4_K_M"])
reply, prompt_tokens, answer_tokens, seconds = eg.ask_gguf(llm, nlu.SHORT_SYSTEM, "turn on the kitchen light")
print(reply)
print(prompt_tokens, "prompt tokens,", answer_tokens, "answer tokens,", round(seconds, 2), "seconds")

**You should see** a correct JSON answer for the kitchen light, from a model running with **no GPU**.

Now the same 24 messages for every size. This takes a few minutes. **Predict.** Write on paper: which format will be the fastest, and will the smallest one (Q2_K) keep its accuracy? **A sample of 24 is small:** one message is about 4 points, so differences of a few points between two sizes are noise, not a result.

In [ ]:
cpu_rows = []
for name in files:
    eg.warm_up(files[name])
    llm = eg.load_gguf(files[name])
    result = eg.evaluate_gguf(llm, nlu.SHORT_SYSTEM, sample, max_seconds=90)
    cpu_rows.append({"format": name, "exact %": result["exact %"], "valid JSON %": result["valid JSON %"],
                     "ms per message": result["ms per message"], "tokens per second": result["tokens per second"],
                     "messages done": result["messages done"]})
    del llm
cpu_table = pd.DataFrame(cpu_rows)
cpu_table

**You should see** the same accuracy for the bigger sizes, and a **cliff** at the smallest: `Q2_K` loses accuracy and often stops producing valid JSON. In our test run: F16, Q8_0, Q5_K_M and Q4_K_M all got 83.3 per cent exact; Q2_K got 45.8 per cent exact and 87.5 per cent valid JSON. Speed: about 6.3 tokens per second for F16 and 11.4 for Q4_K_M.

*If it looks different:* the timing depends on your CPU, and with 24 messages one message is about 4 points, so differences of a few points are noise. The cliff at Q2_K should still be clear.

**The size is not the whole story.** A format that is small and wrong is not an optimisation.

## 18.6 Perplexity is not accuracy
**Perplexity** measures how surprised the model is by the correct answer: 1 means never surprised, and higher means more surprised. It is quick to compute and many people use it to compare quantised models. Let us compute it on 10 validation examples for each size, and put it next to the accuracy.

In [ ]:
validation = nlu.make_training_examples(40, seed=99)[:10]
perplexity_rows = []
for name in files:
    llm = eg.load_gguf(files[name], with_scores=True)
    perplexity = eg.answer_perplexity(llm, nlu.SHORT_SYSTEM, validation)
    perplexity_rows.append({"format": name, "perplexity": perplexity})
    del llm
perplexity_by_format = {}
for row in perplexity_rows:
    perplexity_by_format[row["format"]] = row["perplexity"]
merged_rows = []
for row in cpu_rows:
    merged_rows.append({"format": row["format"], "perplexity": perplexity_by_format[row["format"]],
                        "exact %": row["exact %"], "valid JSON %": row["valid JSON %"]})
pd.DataFrame(merged_rows)

In [ ]:
check(len(merged_rows) == 5, "Checkpoint: perplexity and accuracy are side by side for 5 formats.",
      "Run the CPU table cell of 18.5 first, then this cell again.")

**You should see** perplexity stay close to 1 for the bigger sizes and rise at the smallest (in our test run: 1.001 for the first four, 1.154 for Q2_K). **It does not tell you how many answers are right**: "less surprised by the text" is not the same as "correct". **Perplexity is a quick warning light. The decision is made on the task test.**

## 18.7 The same idea inside PyTorch
The Hugging Face library can also load a model in **8 bits or 4 bits** (`bitsandbytes`), on the GPU. This is what Lab 17 used for training, so it is a second way to do the same idea. We test the merged model at 16, 8 and 4 bits: memory, speed and accuracy, on the 72 messages.

In [ ]:
torch_rows = []
for bits in [16, 8, 4]:
    tok3, model3 = el.load_edge_model(MERGED_FOLDER, bits=bits)
    result = el.evaluate(tok3, model3, nlu.SHORT_SYSTEM)
    torch_rows.append({"bits": bits, "weights MB": el.model_size_mb(model3), "exact %": result["exact %"], "ms per message": result["ms per message"]})
    del model3
    torch.cuda.empty_cache()
pd.DataFrame(torch_rows)

In [ ]:
check(len(torch_rows) == 3, "Checkpoint: you tested 16, 8 and 4 bits in PyTorch.",
      "Run the cell above again. If it stops with an error about memory, restart the kernel and run all cells.")

**You should see** the weights shrink with the bits. The speed on a GPU does not always improve: the saving is mostly **memory**. Quantising is a trade, not a gift. Write down the weights in MB for 16 and 4 bits.

## 18.8 Your turn: choose a format
Your brief (Lab 16) says the accuracy must be at least `BRIEF["accuracy_percent"]` per cent and the model should be as small as possible. Write a loop that goes through the table of GGUF sizes and finds the **smallest format** that meets the accuracy. If none does, the answer is `"none"`.

In [ ]:
BRIEF = {"accuracy_percent": 90}                # 🔧 the target from Lab 16
sizes_by_name = {}
for row in sizes:
    sizes_by_name[row["format"]] = row["size MB"]

In [ ]:
chosen = "F16"                                  # TODO: find the smallest format whose exact % is at least the target
for row in cpu_rows:
    pass                                        # TODO: compare row["exact %"] with BRIEF["accuracy_percent"] and sizes_by_name[row["format"]]
print("chosen format:", chosen)

In [ ]:
check(chosen in files or chosen == "none", "The loop gives a format of the table (or none if no format meets the target).",
      "The answer must be one of the names in the table, or the word none.")

If the answer is `none`, that is a real finding: the brief and the model do not fit yet, and the next steps are in Labs 19 and 20 (and a better dataset). **An engineer reports that honestly.**

## 18.9 What you proved: the third row

In [ ]:
best = cpu_rows[0]
for row in cpu_rows:
    if row["exact %"] >= best["exact %"] - 3 and sizes_by_name[row["format"]] < sizes_by_name[best["format"]]:
        best = row                              # the smallest format that loses at most 3 points of accuracy
el.save_row("3 quantised", {"what": "GGUF " + best["format"] + ", CPU only", "exact_percent": best["exact %"],
                           "ms_per_message": best["ms per message"], "file_mb": sizes_by_name[best["format"]]})
board = el.load_scoreboard()
for step in board:
    print(step, board[step])
check(len(board) >= 3, "Checkpoint: the scoreboard has three rows (prompt only, fine-tuned, quantised).",
      "Run the cell above again. If rows 1 or 2 are missing, run Labs 16 and 17 first.")

**You should see** three rows: prompt only, fine-tuned, and quantised. The quantised row has a **file size** of a fraction of the first rows and runs with **no GPU**.

## What you take away
- Quantisation rounds the weights to fewer levels. The file size follows the bits (size = weights × bits ÷ 8), and you can predict it.
- In our test run, Q4_K_M kept the accuracy of the 16-bit file (83.3 per cent on the sample) at 986 MB instead of 3,094 MB, but Q2_K fell to 45.8 per cent: there is a cliff.
- Merge the adapter first, test the merged model, and decide on the **task** result, not on perplexity alone. Pick the smallest format that still meets the brief.

## Check your understanding
1. About how big is a 0.5-billion-parameter model stored in 16 bits? And in 4 bits?
*✍️ Your answer:* 
2. Why do we merge the adapter before quantising and serving?
*✍️ Your answer:* 
3. Perplexity went up only a little for a format, but accuracy fell a lot. What do you conclude?
*✍️ Your answer:* 
4. Which format would you ship, and why?
*✍️ Your answer:* 

## Next
**Lab 19:** the file is small. Now many people talk to the device at once: how fast, and how many?